# Wan 2.1 Studio · 35mm Video Synthesis on Free Colab (Tesla T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/wan2.1-t4-colab/blob/main/wan2_1_t4_colab_free_inference.ipynb)

A hardware-tailored pipeline running Alibaba's **Wan 2.1 (T2V-1.3B)** with **Native FP16 DiT**, **4-bit NF4 UMT5**, **Step-by-Step Storyboard Keyframe Previews**, and **Multi-Shot 720p HD Reel Assembly** on a **100% free Google Colab instance** (NVIDIA Tesla T4 GPU, 15 GB VRAM).

**Quick Start:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run **Step 1 (Setup)** and **Step 4 (Launch Video Studio)** to control and load everything directly inside the Gradio web app, or use `Runtime → Run all`.
3. Open the public `gradio.live` link printed in Step 4.

<sub>Architecture: Native FP16 DiT (2.6 GB) · 4-bit NF4 UMT5 (2.4 GB) · Official UniPC (flow_shift=5.0) · Dynamic CFG Annealing (5.2→4.4) · Pre-VAE RAM Eviction · Progressive Keyframe Storyboard · Code: <a href="https://github.com/rivas-vega/wan2.1-t4-colab">github.com/rivas-vega/wan2.1-t4-colab</a></sub>

In [ ]:
# @title **1 · Setup Environment** — verify T4 GPU and install dependencies (~1 min)
import os, sys, subprocess

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return r.stdout

try:
    gpu = _run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).strip()
    print(f"GPU detected: {gpu}")
except Exception:
    raise SystemExit("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then run again.")

# Prevent Colab Host RAM OOM during safetensors download: disable memory-buffered Xet reconstruction
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

if os.path.isfile("pipeline_utils.py"):
    REPO_DIR = os.getcwd()
elif os.path.isdir("/content/wan2.1-t4-colab"):
    REPO_DIR = "/content/wan2.1-t4-colab"
    try:
        _run(["git", "-C", REPO_DIR, "pull", "-q"])
    except Exception:
        pass
else:
    REPO_DIR = "/content/wan2.1-t4-colab"
    _run(["git", "clone", "-q", "--depth", "1", "https://github.com/rivas-vega/wan2.1-t4-colab.git", REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # Colab's torchao breaks diffusers imports
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "diffusers>=0.33.0", "transformers>=4.48.0", "accelerate>=0.34.0", "bitsandbytes",
      "gradio", "imageio[ffmpeg]", "ftfy", "sentencepiece", "protobuf", "scipy", "psutil", "Pillow"])

import importlib
for _m in ("pipeline_utils", "app"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
print("Environment Ready.")

In [ ]:
# @title **2 · Hugging Face Token** *(optional — can also be set inside the Gradio Studio)*
# @markdown Free read token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Enables Qwen 2.5-72B Cloud Prompt Director and faster weight streaming.
HF_TOKEN = ""  # @param {type:"string"}

import os
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception:
    pass

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("Token configured. Authenticated streaming and Qwen 2.5-72B Director enabled.")
else:
    print("No token set. Using anonymous streaming and built-in 35mm Documentary Prompt Director.")

In [ ]:
# @title **3 · Preload Wan 2.1 Engine** *(optional — Gradio Studio also auto-loads on demand)*
import os, sys, importlib, warnings, psutil
warnings.filterwarnings("ignore")

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

from diffusers.utils import logging as _dl
from transformers.utils import logging as _tl
_dl.set_verbosity_error(); _tl.set_verbosity_error()

import pipeline_utils
importlib.reload(pipeline_utils)
from pipeline_utils import load_wan_t4_pipeline, clear_cuda_cache

print("Initializing Wan 2.1 Engine (Native FP16 DiT + 4-bit NF4 UMT5 for 15GB T4 GPU / 12.7GB RAM)...")
pipe = load_wan_t4_pipeline(quantization="fast_fp16")
clear_cuda_cache()

ram_used = (psutil.virtual_memory().total - psutil.virtual_memory().available) / 1e9
ram_total = psutil.virtual_memory().total / 1e9
ram_margin = ram_total - ram_used
vram_used = 0.0
try:
    import torch
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_allocated() / 1e9
except Exception:
    pass

print(f"\nEngine Ready | Host RAM: {ram_used:.1f}/{ram_total:.1f} GB ({ram_margin:.1f} GB free) | Static VRAM: {vram_used:.2f} GB")

In [ ]:
# @title **4 · Launch Wan 2.1 Studio** — open the public Gradio link below
# @markdown Launches the progressive web studio. If Step 3 was skipped, the studio will initialize the engine automatically inside the web UI.
import importlib, pipeline_utils, app
importlib.reload(pipeline_utils); importlib.reload(app)
from app import build_demo, launch

demo = build_demo(pipe=globals().get("pipe"), output_dir="/content/outputs")
launch(demo, share=True, debug=False)

In [ ]:
# @title **5 · Direct Notebook Runner** *(optional inline generation without Gradio)*
PROMPT = "A matte-black sports car driving smoothly along a wet Tokyo street at night under soft streetlights"  # @param {type:"string"}
PRESET = "Reel 9:16 · Single Shot"  # @param ["Reel 9:16 · Single Shot", "Reel 9:16 · 3-Shot Story", "Cinema 16:9 · Widescreen"]

import importlib, pipeline_utils
importlib.reload(pipeline_utils)
from IPython.display import Video, display
from pipeline_utils import create_clip

path, seed, secs, _ = create_clip(globals().get("pipe"), prompt=PROMPT, duration=PRESET, output_dir="/content/outputs")
print(f"Mastered in {secs / 60:.1f} min · Seed {seed} · {path}")
display(Video(path, embed=True, width=380, html_attributes="controls autoplay loop"))